# Native IWR6843 UAV micro-Doppler CNN

This notebook trains the CNN used by `inference.py` from two labelled datasets: `dataset/uav/` (positive) and `dataset/others/` (negative). Consecutive feature steps are divided into non-overlapping 48-step windows, then the individual windows are split 70%/15%/15% into train, validation, and test with class stratification.

The input contract is `[2, 48, 64]`: target-gate log power and target-minus-local-background log power. The current live feature extractor averages **three** target range bins. All training features must be captured with this current three-bin implementation.

Accepted files are `.npz`, `.npy`, and JSONL containing `classification_feature`. Arrays may contain complete windows `[N,2,48,64]`, one window `[2,48,64]`, or feature steps `[T,2,64]`. NPZ files should use the key `windows` or `feature_steps` and should include `feature_version`, `target_gate_range_bins`, and `compatible_profile_sha256`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import os
import platform
import random
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    f1_score, precision_recall_curve, precision_score, recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

configured_repo = os.environ.get('RADAR_UREX_REPO')
repo_candidates = ([Path(configured_repo)] if configured_repo else []) + [
    Path.cwd(), Path('/content/Radar-UREx'),
    Path('/content/drive/MyDrive/UREX/Radar-UREx'),
]
REPO = next((path for path in repo_candidates if (path / 'inference.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Radar-UREx repository not found; set RADAR_UREX_REPO.')
REPO = REPO.resolve()
os.chdir(REPO)

from inference import (
    CNN_CONFIG, DEPLOYMENT_STATUS, DOPPLER_BINS, FEATURE_VERSION, INPUT_SHAPE_CHW,
    TARGET_GATE_BINS, WINDOW_STEPS, _build_model,
    normalized_profile_sha256,
)

SEED = 42
WINDOW_STRIDE = WINDOW_STEPS
TRAIN_WINDOW_FRACTION = 0.70
VALIDATION_WINDOW_FRACTION = 0.15
TEST_WINDOW_FRACTION = 0.15
assert np.isclose(TRAIN_WINDOW_FRACTION + VALIDATION_WINDOW_FRACTION + TEST_WINDOW_FRACTION, 1.0)
CLIP_PERCENTILES = (0.5, 99.5)
BATCH_SIZE = 32
EPOCHS = 100
PATIENCE = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MMHAWKEYE_LSTM_PARAMETERS = 231_682
LABEL_TO_INDEX = {'others': 0, 'uav': 1}
DATASET_ROOT = REPO / 'dataset'
CLASS_DIRECTORIES = {name: DATASET_ROOT / name for name in LABEL_TO_INDEX}
PROFILE_PATH = REPO / 'rawdatacapture' / 'profile.cfg'
PROFILE_SHA256 = normalized_profile_sha256(PROFILE_PATH)
OUTPUT_DIR = REPO / 'training_output' / 'micro_doppler_cnn'
EMPTY_DATASET_ROOT = DATASET_ROOT / 'empty'

print({
    'python': platform.python_version(),
    'dataset_root': str(DATASET_ROOT),
    'classes': {name: str(path) for name, path in CLASS_DIRECTORIES.items()},
    'feature_version': FEATURE_VERSION,
    'input_shape_chw': INPUT_SHAPE_CHW,
    'target_gate_range_bins': TARGET_GATE_BINS,
    'cnn_config': CNN_CONFIG,
    'profile_sha256': PROFILE_SHA256,
})
assert TARGET_GATE_BINS == 3


## 1. Load and validate the two class datasets

Folder names provide labels; filenames do not. Source-file identities are retained for manifests and reporting, but the split operates on individual non-overlapping windows. Files without a complete 48-frame tracked-target run are moved to `dataset/empty/<original-class>/`; null no-target frames split valid runs and are never included in a window. Existing files are never overwritten. Malformed files and metadata mismatches are rejected without being moved. JSONL must explicitly contain the same `classification_feature` steps consumed by live inference; display-only Doppler spectra are not silently treated as the two-channel input.

In [ ]:
def scalar_text(value: Any) -> str:
    array = np.asarray(value)
    if array.size != 1:
        raise ValueError(f'Expected scalar metadata, received shape {array.shape}')
    item = array.reshape(()).item()
    return item.decode() if isinstance(item, bytes) else str(item)


def validate_metadata(metadata: dict[str, Any], path: Path) -> None:
    if 'feature_version' in metadata:
        observed = scalar_text(metadata['feature_version'])
        if observed != FEATURE_VERSION:
            raise ValueError(f'{path}: feature_version={observed!r}, expected {FEATURE_VERSION!r}')
    if 'target_gate_range_bins' in metadata:
        observed = int(np.asarray(metadata['target_gate_range_bins']).reshape(()))
        if observed != TARGET_GATE_BINS:
            raise ValueError(f'{path}: target gate uses {observed} bins; expected {TARGET_GATE_BINS}')
    if 'compatible_profile_sha256' in metadata:
        observed = scalar_text(metadata['compatible_profile_sha256'])
        if observed != PROFILE_SHA256:
            raise ValueError(f'{path}: profile fingerprint does not match rawdatacapture/profile.cfg')


def steps_to_windows(steps: np.ndarray) -> np.ndarray:
    steps = np.asarray(steps, dtype=np.float32)
    if steps.ndim != 3 or steps.shape[1:] != (2, DOPPLER_BINS):
        raise ValueError(f'Feature steps must have shape [T,2,{DOPPLER_BINS}], got {steps.shape}')
    starts = range(0, len(steps) - WINDOW_STEPS + 1, WINDOW_STRIDE)
    windows = [steps[start:start + WINDOW_STEPS].transpose(1, 0, 2) for start in starts]
    if not windows:
        return np.empty((0, *INPUT_SHAPE_CHW), dtype=np.float32)
    return np.stack(windows).astype(np.float32)


def array_to_windows(values: np.ndarray, path: Path) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    if values.shape == INPUT_SHAPE_CHW:
        windows = values[None]
    elif values.ndim == 4 and tuple(values.shape[1:]) == INPUT_SHAPE_CHW:
        windows = values
    elif values.ndim == 3 and values.shape[1:] == (2, DOPPLER_BINS):
        windows = steps_to_windows(values)
    else:
        raise ValueError(f'{path}: unsupported feature array shape {values.shape}')
    if not np.isfinite(windows).all():
        raise ValueError(f'{path}: feature data contains non-finite values')
    return np.ascontiguousarray(windows, dtype=np.float32)


def load_np_file(path: Path) -> tuple[np.ndarray, dict[str, Any]]:
    if path.suffix.lower() == '.npy':
        return array_to_windows(np.load(path, allow_pickle=False), path), {}
    with np.load(path, allow_pickle=False) as archive:
        keys = set(archive.files)
        data_key = next((key for key in ('windows', 'feature_steps', 'features') if key in keys), None)
        if data_key is None:
            raise ValueError(f'{path}: NPZ needs windows or feature_steps')
        metadata = {
            key: archive[key]
            for key in ('feature_version', 'target_gate_range_bins', 'compatible_profile_sha256')
            if key in keys
        }
        validate_metadata(metadata, path)
        return array_to_windows(archive[data_key], path), metadata


def load_jsonl(path: Path) -> tuple[np.ndarray, dict[str, Any]]:
    metadata: dict[str, Any] = {}
    runs: list[list[np.ndarray]] = []
    current_run: list[np.ndarray] = []
    previous_frame_index: int | None = None
    with path.open('r', encoding='utf-8') as source:
        for line_number, line in enumerate(source, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f'{path}:{line_number}: invalid JSON') from exc
            if record.get('record_type') == 'metadata':
                if int(record.get('display_update_every', -1)) != 1:
                    raise ValueError(f'{path}: capture needs display_update_every=1 for consecutive 30 Hz features')
                classification = record.get('classification_feature') or {}
                metadata = {
                    key: classification[key]
                    for key in ('feature_version', 'target_gate_range_bins', 'compatible_profile_sha256')
                    if key in classification
                }
                validate_metadata(metadata, path)
                continue
            if record.get('record_type') != 'update':
                continue
            feature = record.get('classification_feature')
            frame_index = int(record.get('processed_frame_index', -1))
            consecutive = previous_frame_index is None or frame_index == previous_frame_index + 1
            if feature is None or not consecutive:
                if current_run:
                    runs.append(current_run)
                current_run = []
            if feature is not None:
                step = np.asarray(feature, dtype=np.float32)
                if step.shape != (2, DOPPLER_BINS) or not np.isfinite(step).all():
                    raise ValueError(f'{path}:{line_number}: invalid classification_feature {step.shape}')
                current_run.append(step)
            previous_frame_index = frame_index
    if current_run:
        runs.append(current_run)
    complete_runs = [np.stack(run) for run in runs if len(run) >= WINDOW_STEPS]
    if not complete_runs:
        return np.empty((0, *INPUT_SHAPE_CHW), dtype=np.float32), metadata
    return np.concatenate([steps_to_windows(run) for run in complete_runs]), metadata


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def load_dataset() -> tuple[np.ndarray, np.ndarray, np.ndarray, list[dict[str, Any]]]:
    all_windows: list[np.ndarray] = []
    all_labels: list[np.ndarray] = []
    all_groups: list[np.ndarray] = []
    manifest: list[dict[str, Any]] = []
    for label_name, directory in CLASS_DIRECTORIES.items():
        if not directory.is_dir():
            raise FileNotFoundError(f'Missing class directory: {directory}')
        paths = sorted(path for path in directory.rglob('*') if path.suffix.lower() in {'.npz', '.npy', '.jsonl'})
        if not paths:
            raise FileNotFoundError(f'No supported feature files under {directory}')
        for path in paths:
            windows, metadata = load_jsonl(path) if path.suffix.lower() == '.jsonl' else load_np_file(path)
            group = path.relative_to(DATASET_ROOT).as_posix()
            if len(windows) == 0:
                destination = EMPTY_DATASET_ROOT / group
                destination.parent.mkdir(parents=True, exist_ok=True)
                if destination.exists():
                    raise FileExistsError(f'Refusing to overwrite existing empty capture: {destination}')
                path.rename(destination)
                print(f'Moved {group} to {destination.relative_to(DATASET_ROOT).as_posix()}: no complete {WINDOW_STEPS}-frame tracked-target window')
                continue
            all_windows.append(windows)
            all_labels.append(np.full(len(windows), LABEL_TO_INDEX[label_name], dtype=np.int64))
            all_groups.append(np.full(len(windows), group, dtype=object))
            manifest.append({
                'path': group, 'sha256': file_sha256(path),
                'label': label_name, 'windows': len(windows),
                'metadata_fields': sorted(metadata),
            })
    if not all_windows:
        raise ValueError(f'No files contain a complete {WINDOW_STEPS}-frame tracked-target window')
    windows = np.concatenate(all_windows).astype(np.float32)
    labels = np.concatenate(all_labels)
    groups = np.concatenate(all_groups)
    if set(labels.tolist()) != set(LABEL_TO_INDEX.values()):
        raise ValueError('Both dataset/uav and dataset/others need usable windows')
    return windows, labels, groups, manifest


## 2. Stratified window split and train-only normalization

Non-overlapping windows are independently split 70%/15%/15%, stratified by class and reproducible from the fixed seed. A source recording can therefore contribute windows to every partition. Clipping limits, channel means, and channel standard deviations are fitted only on training windows. Class imbalance is handled by the training loss rather than discarding captures.

In [ ]:
def stratified_window_split(labels: np.ndarray) -> dict[str, np.ndarray]:
    indices = np.arange(len(labels))
    train_indices, holdout_indices = train_test_split(
        indices,
        train_size=TRAIN_WINDOW_FRACTION,
        random_state=SEED,
        shuffle=True,
        stratify=labels,
    )
    relative_test_fraction = TEST_WINDOW_FRACTION / (VALIDATION_WINDOW_FRACTION + TEST_WINDOW_FRACTION)
    validation_indices, test_indices = train_test_split(
        holdout_indices,
        test_size=relative_test_fraction,
        random_state=SEED,
        shuffle=True,
        stratify=labels[holdout_indices],
    )
    splits = {
        'train': np.sort(train_indices),
        'validation': np.sort(validation_indices),
        'test': np.sort(test_indices),
    }
    expected_labels = set(LABEL_TO_INDEX.values())
    for name, split_indices in splits.items():
        if set(np.unique(labels[split_indices]).tolist()) != expected_labels:
            raise ValueError(f'{name} split does not contain both classes')
    combined = np.concatenate(list(splits.values()))
    assert len(combined) == len(indices)
    assert len(np.unique(combined)) == len(indices)
    return splits


def fit_normalization(training_windows: np.ndarray) -> dict[str, np.ndarray]:
    axes = (0, 2, 3)
    clip_low = np.percentile(training_windows, CLIP_PERCENTILES[0], axis=axes).astype(np.float32)
    clip_high = np.percentile(training_windows, CLIP_PERCENTILES[1], axis=axes).astype(np.float32)
    clipped = np.clip(training_windows, clip_low[None, :, None, None], clip_high[None, :, None, None])
    channel_mean = clipped.mean(axis=axes, dtype=np.float64).astype(np.float32)
    channel_std = clipped.std(axis=axes, dtype=np.float64).astype(np.float32)
    if np.any(channel_std <= np.finfo(np.float32).eps):
        raise ValueError('A feature channel has zero training variance')
    return {'clip_low': clip_low, 'clip_high': clip_high, 'channel_mean': channel_mean, 'channel_std': channel_std}


def normalize(windows: np.ndarray, stats: dict[str, np.ndarray]) -> np.ndarray:
    clipped = np.clip(windows, stats['clip_low'][None, :, None, None], stats['clip_high'][None, :, None, None])
    result = (clipped - stats['channel_mean'][None, :, None, None]) / stats['channel_std'][None, :, None, None]
    if not np.isfinite(result).all():
        raise ValueError('Normalization produced non-finite values')
    return np.ascontiguousarray(result, dtype=np.float32)


raw_windows, labels, groups, dataset_manifest = load_dataset()
splits = stratified_window_split(labels)
normalization = fit_normalization(raw_windows[splits['train']])
windows = normalize(raw_windows, normalization)
manifest_sha256 = hashlib.sha256(json.dumps(dataset_manifest, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
split_summary = {
    name: {
        'windows': len(indices),
        'captures': len(set(groups[indices].tolist())),
        **{label: int(np.sum(labels[indices] == index)) for label, index in LABEL_TO_INDEX.items()},
    }
    for name, indices in splits.items()
}
print(json.dumps({'manifest_sha256': manifest_sha256, 'files': dataset_manifest, 'splits': split_summary}, indent=2))
print('normalization:', {key: value.tolist() for key, value in normalization.items()})


## 3. Larger depthwise-residual CNN

The CNN uses the same implementation and configuration as live inference. Its approximately 230k parameters closely match the 231,682-parameter two-layer LSTM in `mmHawkeye--`, while retaining convolutional micro-Doppler processing.

In [ ]:
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, Dataset
except ModuleNotFoundError as exc:
    raise RuntimeError('Install PyTorch >=2.6,<3 to train the CNN') from exc

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class RadarWindowDataset(Dataset):
    def __init__(self, values: np.ndarray, targets: np.ndarray, augment: bool = False):
        self.values = values
        self.targets = targets.astype(np.float32)
        self.augment = augment

    def __len__(self) -> int:
        return len(self.targets)

    def __getitem__(self, index: int):
        value = torch.from_numpy(self.values[index].copy())
        if self.augment:
            value += torch.randn_like(value) * 0.01
            if torch.rand(()) < 0.5:
                shift = int(torch.randint(-2, 3, ()).item())
                shifted = torch.zeros_like(value)
                if shift > 0:
                    shifted[..., shift:] = value[..., :-shift]
                elif shift < 0:
                    shifted[..., :shift] = value[..., -shift:]
                else:
                    shifted = value
                value = shifted
        return value, torch.tensor(self.targets[index], dtype=torch.float32)


def make_loader(split: str, *, shuffle: bool, augment: bool = False) -> DataLoader:
    indices = splits[split]
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        RadarWindowDataset(windows[indices], labels[indices], augment=augment),
        batch_size=BATCH_SIZE if split == 'train' else 64,
        shuffle=shuffle, num_workers=0,
        generator=generator if shuffle else None,
    )


train_loader = make_loader('train', shuffle=True, augment=True)
validation_loader = make_loader('validation', shuffle=False)
test_loader = make_loader('test', shuffle=False)
model = _build_model(nn, CNN_CONFIG).to(DEVICE)
cnn_parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
relative_parameter_difference = abs(cnn_parameter_count - MMHAWKEYE_LSTM_PARAMETERS) / MMHAWKEYE_LSTM_PARAMETERS
assert relative_parameter_difference < 0.05
print(f'CNN parameters: {cnn_parameter_count:,}; mmHawkeye LSTM: {MMHAWKEYE_LSTM_PARAMETERS:,}; difference: {relative_parameter_difference:.2%}')

negative_count = float(np.sum(labels[splits['train']] == 0))
positive_count = float(np.sum(labels[splits['train']] == 1))
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([negative_count / positive_count], device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


def collect_logits(loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    logits, targets = [], []
    with torch.inference_mode():
        for batch_values, batch_targets in loader:
            logits.append(model(batch_values.to(DEVICE)).cpu().numpy())
            targets.append(batch_targets.numpy())
    return np.concatenate(logits), np.concatenate(targets).astype(np.int8)


best_state = None
best_validation_pr_auc = -np.inf
best_epoch = -1
epochs_without_improvement = 0
training_history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for batch_values, batch_targets in train_loader:
        batch_values = batch_values.to(DEVICE)
        batch_targets = batch_targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(batch_values), batch_targets)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += float(loss.detach()) * len(batch_targets)
    validation_logits, validation_targets = collect_logits(validation_loader)
    validation_pr_auc = average_precision_score(validation_targets, validation_logits)
    epoch_loss = total_loss / len(splits['train'])
    training_history.append({'epoch': epoch, 'loss': epoch_loss, 'validation_pr_auc': validation_pr_auc})
    if validation_pr_auc > best_validation_pr_auc + 1e-4:
        best_validation_pr_auc = float(validation_pr_auc)
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    print(f'epoch={epoch:03d} loss={epoch_loss:.5f} validation_PR-AUC={validation_pr_auc:.5f}')
    if epochs_without_improvement >= PATIENCE:
        break
if best_state is None:
    raise RuntimeError('Training did not produce a checkpoint')
model.load_state_dict(best_state)
model.eval()
plt.plot([row['epoch'] for row in training_history], [row['loss'] for row in training_history])
plt.xlabel('epoch'); plt.ylabel('training loss'); plt.grid(alpha=0.25); plt.show()


## 4. Calibrate, evaluate, and export

Validation logits fit the probability calibrator and select the F1 threshold. The held-out test windows are touched only afterward. Because the split is window-level, recordings may still be represented in more than one partition. Artifacts use the exact filenames loaded by `inference.py`; exporting intentionally replaces the obsolete five-bin model bundle.

In [ ]:
%pip install onnx

def metric_summary(truth: np.ndarray, probability: np.ndarray, threshold: float) -> dict[str, Any]:
    predicted = (probability >= threshold).astype(np.int8)
    return {
        'samples': int(len(truth)), 'threshold': float(threshold),
        'recall': float(recall_score(truth, predicted, zero_division=0)),
        'precision': float(precision_score(truth, predicted, zero_division=0)),
        'f1': float(f1_score(truth, predicted, zero_division=0)),
        'pr_auc': float(average_precision_score(truth, probability)),
        'roc_auc': float(roc_auc_score(truth, probability)),
        'brier_score': float(brier_score_loss(truth, probability)),
        'confusion_matrix_tn_fp_fn_tp': confusion_matrix(truth, predicted, labels=[0, 1]).ravel().tolist(),
    }


validation_logits, validation_targets = collect_logits(validation_loader)
calibrator = LogisticRegression(C=1e3, solver='lbfgs', max_iter=500, random_state=SEED)
calibrator.fit(validation_logits.reshape(-1, 1), validation_targets)
validation_probability = calibrator.predict_proba(validation_logits.reshape(-1, 1))[:, 1]
precision, recall, thresholds = precision_recall_curve(validation_targets, validation_probability)
f1_values = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
threshold = float(thresholds[int(np.argmax(f1_values))])

test_logits, test_targets = collect_logits(test_loader)
test_probability = calibrator.predict_proba(test_logits.reshape(-1, 1))[:, 1]
window_test_metrics = metric_summary(test_targets, test_probability, threshold)
test_indices = splits['test']
source_truth, source_probability = [], []
for group in np.unique(groups[test_indices]):
    mask = groups[test_indices] == group
    source_truth.append(int(test_targets[mask][0]))
    source_probability.append(float(test_probability[mask].mean()))
source_aggregated_test_metrics = metric_summary(np.asarray(source_truth), np.asarray(source_probability), threshold)
print('window test metrics:', json.dumps(window_test_metrics, indent=2))
print('source-aggregated test-window metrics:', json.dumps(source_aggregated_test_metrics, indent=2))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
state_path = OUTPUT_DIR / 'model_state.pt'
calibration_path = OUTPUT_DIR / 'calibration.joblib'
card_path = OUTPUT_DIR / 'manifest.json'
onnx_path = OUTPUT_DIR / 'model.onnx'
parity_path = OUTPUT_DIR / 'parity.npz'
try:
    import onnx
except ModuleNotFoundError as exc:
    raise RuntimeError('Install ONNX before exporting: pip install onnx') from exc
model.eval()
onnx_sample = torch.zeros((1, *INPUT_SHAPE_CHW), dtype=torch.float32, device=DEVICE)
torch.onnx.export(
    model, (onnx_sample,), onnx_path,
    input_names=['input'], output_names=['logit'],
    opset_version=18, dynamo=False,
)
onnx.checker.check_model(onnx.load(onnx_path))
parity_count = min(128, len(test_indices))
parity_positions = np.linspace(0, len(test_indices) - 1, parity_count, dtype=np.int64)
np.savez_compressed(
    parity_path,
    normalized_windows=windows[test_indices[parity_positions]],
    probabilities=test_probability[parity_positions],
    feature_version=np.asarray(FEATURE_VERSION),
)
torch.save({
    'state_dict': {key: value.detach().cpu() for key, value in model.state_dict().items()},
    'architecture': 'MicroDopplerCNN', 'cnn_config': CNN_CONFIG,
    'input_shape_chw': INPUT_SHAPE_CHW, 'feature_version': FEATURE_VERSION,
    'target_gate_range_bins': TARGET_GATE_BINS,
}, state_path)
joblib.dump({
    'calibrator': calibrator, 'threshold': threshold,
    **normalization, 'compatible_profile_sha256': PROFILE_SHA256,
    'dataset_manifest_sha256': manifest_sha256,
}, calibration_path, compress=3)
model_card = {
    'architecture': 'depthwise-separable residual 2D CNN',
    'parameters': cnn_parameter_count, 'mmhawkeye_lstm_parameters': MMHAWKEYE_LSTM_PARAMETERS,
    'best_epoch': best_epoch, 'validation_pr_auc': best_validation_pr_auc,
    'threshold': threshold, 'window_test_metrics': window_test_metrics,
    'source_aggregated_test_window_metrics': source_aggregated_test_metrics,
    'feature_version': FEATURE_VERSION, 'input_shape_chw': list(INPUT_SHAPE_CHW),
    'target_gate_range_bins': TARGET_GATE_BINS,
    'classes': {'positive': 'dataset/uav', 'negative': 'dataset/others'},
    'split_policy': 'non-overlapping windows, stratified 70/15/15 by class',
    'split_fractions': {'train': TRAIN_WINDOW_FRACTION, 'validation': VALIDATION_WINDOW_FRACTION, 'test': TEST_WINDOW_FRACTION},
    'dataset_manifest_sha256': manifest_sha256,
    'compatible_profile_sha256': PROFILE_SHA256,
    'onnx_sha256': file_sha256(onnx_path),
    'parity_data_sha256': file_sha256(parity_path),
    'deployment_backend': 'TensorRT FP16 on Jetson; PyTorch CPU fallback',
    'deployment_status': DEPLOYMENT_STATUS,
}
card_path.write_text(json.dumps(model_card, indent=2), encoding='utf-8')
print('saved:', state_path, calibration_path, card_path, onnx_path, parity_path)
